# 1.1 — Ingestão de ordens de venda

- **Propósito:** Ingerir incrementalmente arquivos TXT do SAP e manter a tabela `raw_sales_order`.
- **Entrada:** Arquivos TXT delimitados por pipe
- **Saída:** `parts_hdbk_sandbox.dt_sales_orders.raw_sales_order`
- **Chave:** numero_ov + item · **Carga:** Incremental via Auto Loader

In [0]:
from delta.tables import DeltaTable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

from typing import Sequence

import uuid

In [0]:
# Constantes do Pipeline

# -----------------------------------------------------------------------------
# Layout do arquivo SAP
# -----------------------------------------------------------------------------

SALES_ORDER_LAYOUT = [
    "numero_ov",
    "data",
    "tipo_ov",
    "motivo_ov",
    "bloqueio_rem",
    "motivo_recusa",
    "org_vendas",
    "canal_dist",
    "setor_ativ",
    "centro",
    "emissor_da_ordem",
    "autor",
    "item",
    "material",
    "categoria_do_item",
    "item_superior",
    "quantidade",
    "um"
]

RAW_BUSINESS_COLUMNS = [
    "numero_ov", "data", "tipo_ov", "motivo_ov", "bloqueio_rem",
    "motivo_recusa", "org_vendas", "canal_dist", "setor_ativ", "centro",
    "emissor_da_ordem", "numero_pedido", "autor", "item", "material",
    "categoria_do_item", "item_superior", "quantidade", "um",
]
SOURCE_METADATA_COLUMNS = [
    "_source_file_name", "_source_file_path", "_source_file_modification_time",
]
AUDIT_COLUMNS = [
    "_ingested_at", "_last_updated_at", "_ingested_by", "_load_type", "_load_id",
]
RAW_COLUMNS = RAW_BUSINESS_COLUMNS + SOURCE_METADATA_COLUMNS + AUDIT_COLUMNS

# -----------------------------------------------------------------------------
# Chave de Negócio
# -----------------------------------------------------------------------------

BUSINESS_KEY = [
    "numero_ov",
    "item"
]

# -----------------------------------------------------------------------------
# Colunas para Padronização
# -----------------------------------------------------------------------------

DATE_COLUMNS = [
    "data"
]

NUMERIC_COLUMNS = [
    "quantidade"
]


In [0]:
# Configuração do Catalog, Schema e Tabelas

# -----------------------------------------------------------------------------
# Catálogo
# -----------------------------------------------------------------------------

CATALOG = "parts_hdbk_sandbox"
SCHEMA = "dt_sales_orders"

# -----------------------------------------------------------------------------
# Tabelas
# -----------------------------------------------------------------------------

RAW_TABLE = f"{CATALOG}.{SCHEMA}.raw_sales_order"

# -----------------------------------------------------------------------------
# Caminhos
# -----------------------------------------------------------------------------

VOLUME_PATH = "/Volumes/parts_hdbk_sandbox/dt_sales_orders/sap_sales_order"

INPUT_PATH = f"{VOLUME_PATH}/incremental"

CHECKPOINT_PATH = f"{VOLUME_PATH}/_checkpoint/raw_sales_order"

SCHEMA_PATH = f"{VOLUME_PATH}/_schema/raw_sales_order"

# -----------------------------------------------------------------------------
# Auto Loader
# -----------------------------------------------------------------------------

AUTO_LOADER_FORMAT = "cloudFiles"
FILE_FORMAT = "text"
FILE_PATTERN = "*.txt"
INCLUDE_EXISTING_FILES = True

# -----------------------------------------------------------------------------
# Streaming
# -----------------------------------------------------------------------------

TRIGGER_AVAILABLE_NOW = True

# -----------------------------------------------------------------------------
# Modo de Carga
# -----------------------------------------------------------------------------
LOAD_MODE = "INCREMENTAL"
# Opções:
# "INCREMENTAL"
# "period_refresh"

LOAD_ID = str(uuid.uuid4())
CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]

SOURCE_METADATA_SCHEMA = {
    "_source_file_name": "STRING",
    "_source_file_path": "STRING",
    "_source_file_modification_time": "TIMESTAMP",
}

In [0]:
# Função de leitura incremental utilizando Auto Loader
# Configura schema inference e leitura de arquivos TXT delimitados por pipe

def read_sales_order() -> DataFrame:
    """
    Realiza a leitura incremental dos arquivos TXT exportados do SAP
    utilizando Auto Loader.

    Returns
    -------
    DataFrame
        Streaming DataFrame contendo uma linha por registro do arquivo.
    """

    reader = (
        spark.readStream
            .format(AUTO_LOADER_FORMAT)
            .option("cloudFiles.format", FILE_FORMAT)
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .option(
                "cloudFiles.includeExistingFiles",
                str(INCLUDE_EXISTING_FILES).lower()
            )
            .option("pathGlobFilter", FILE_PATTERN)
    )

    return reader.load(INPUT_PATH)

In [0]:
# Função para iniciar o processamento em streaming
# Utiliza foreachBatch para processar micro-batches

def start_stream(df: DataFrame) -> None:
    """
    Inicia o processamento do pipeline utilizando Structured Streaming.
    """

    (
        df.writeStream
            .trigger(availableNow=TRIGGER_AVAILABLE_NOW)
            .option("checkpointLocation", CHECKPOINT_PATH)
            .foreachBatch(process_batch)
            .start()
            .awaitTermination()
    )

In [0]:
def process_batch(df: DataFrame, batch_id: int) -> None:
    """
    Processa um micro-batch recebido pelo Auto Loader.
    
    Etapas:
    1. Parse do arquivo SAP utilizando layout configurado
    2. Padronização de tipos de dados e formatos
    3. Deduplicação dentro do batch
    4. Persistência via MERGE na tabela Delta
    5. Log de métricas do batch processado
    """
    parsed_df = parse_sap_file(
        df,
        SALES_ORDER_LAYOUT
    )
    standardized_df = standardize_sales_order(
        parsed_df
    )

    deduplicated_df = deduplicate_batch(
        standardized_df
    )

    persist_sales_order(
        deduplicated_df
    )

    log_batch_metrics(batch_id)

In [0]:
# Função de parser para arquivos TXT exportados do SAP
# Divide a coluna value utilizando delimitador e aplica o layout configurado

def parse_sap_file(
    df: DataFrame,
    layout: Sequence[str]
) -> DataFrame:
    """
    Realiza o parse de arquivos TXT exportados do SAP.
    """

    expected_columns = len(layout)

    parsed_df = (
        df
        .filter(F.trim(F.col("value")) != "")
        .withColumn(
            "value",
            F.regexp_replace(F.col("value"), r"^\|", "")
        )
        .withColumn(
            "value",
            F.regexp_replace(F.col("value"), r"\|$", "")
        )
        .withColumn(
            "fields",
            F.split(F.col("value"), r"\|")
        )
        .filter(
            F.size(F.col("fields")) == expected_columns
        )
        .filter(
            ~F.col("fields")[0].startswith("Doc.")
        )
        .filter(
            F.trim(F.col("fields")[1]).rlike(r"^\d{2}\.\d{2}\.\d{4}$")
        )
    )

    return parsed_df.select(
    "_source_file_name",
    "_source_file_path",
    "_source_file_modification_time",
    *[
        F.trim(F.col("fields")[i]).alias(column)
        for i, column in enumerate(layout)
    ]
)

In [0]:
# Função de padronização de tipos de dados
# Converte tipos, formata datas e adiciona colunas de auditoria

def standardize_sales_order(df: DataFrame) -> DataFrame:
    """
    Padroniza os dados do relatório de Sales Order.
    """

    df = standardize_nulls(df)
    df = standardize_dates(df)
    df = standardize_numbers(df)

    df = (
        df
        .withColumn(
            "numero_pedido",
            F.lit(None).cast("string")
        )
    )

    df = add_metadata(df)

    return df.select(*RAW_COLUMNS)

def standardize_dates(df: DataFrame) -> DataFrame:
    """
    Converte colunas de data para o tipo Date.
    """

    for column in DATE_COLUMNS:
        df = df.withColumn(
            column,
            F.to_date(F.col(column), "dd.MM.yyyy")
        )

    return df


def standardize_numbers(df: DataFrame) -> DataFrame:
    """
    Converte colunas numéricas para Double.
    """

    for column in NUMERIC_COLUMNS:
        raw_value = F.trim(F.col(column))
        signed_value = F.when(
            raw_value.rlike(r".*-$"),
            F.concat(F.lit("-"), F.regexp_replace(raw_value, r"-$", "")),
        ).otherwise(raw_value)
        normalized_value = F.regexp_replace(
            F.regexp_replace(signed_value, r"\.", ""),
            ",",
            ".",
        )
        df = df.withColumn(column, normalized_value.cast("double"))

    return df


def standardize_nulls(df: DataFrame) -> DataFrame:
    """
    Converte strings vazias em NULL.
    """

    for column in df.schema.fieldNames():
        if not isinstance(df.schema[column].dataType, T.StringType):
            continue
        df = df.withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "",
                F.lit(None)
            ).otherwise(F.col(column))
        )

    return df

def add_metadata(df: DataFrame) -> DataFrame:
    """
    Adiciona metadados de ingestão.
    """

    return df.withColumns({
        "_ingested_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_last_updated_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_ingested_by": F.lit(CURRENT_USER),
        "_load_type": F.lit(LOAD_MODE),
        "_load_id": F.lit(LOAD_ID),
    })

In [0]:
# Função de deduplicação dentro do micro-batch
# Remove registros duplicados utilizando a chave de negócio definida

def deduplicate_batch(df: DataFrame) -> DataFrame:
    """
    Remove registros duplicados dentro do micro-batch.
    """

    return df.dropDuplicates(BUSINESS_KEY)

In [0]:
# Funções de persistência na tabela Delta
# Realiza MERGE incremental utilizando a chave de negócio

def ensure_raw_table_schema() -> None:
    """Migra o metadado legado e garante o schema de rastreabilidade da RAW."""

    if not spark.catalog.tableExists(RAW_TABLE):
        raise RuntimeError(f"Tabela RAW não encontrada: {RAW_TABLE}. Execute o notebook 00 primeiro.")

    existing_columns = set([field.name for field in spark.table(RAW_TABLE).schema.fields])
    missing_columns = [
        f"{name} {data_type}"
        for name, data_type in SOURCE_METADATA_SCHEMA.items()
        if name not in existing_columns
    ]
    if missing_columns:
        spark.sql(f"ALTER TABLE {RAW_TABLE} ADD COLUMNS ({', '.join(missing_columns)})")


def persist_sales_order(df: DataFrame) -> None:
    """
    Persiste os dados na tabela RAW conforme o modo de carga.
    """

    if LOAD_MODE == "PERIOD_REFRESH":
        period_refresh(df)
    else:
        incremental_load(df)

### INCREMENTAL LOAD

def incremental_load(df: DataFrame) -> None:
    """
    Realiza carga incremental utilizando MERGE.
    """

    target = DeltaTable.forName(spark, RAW_TABLE)

    merge_condition = " AND ".join(
        [f"t.{c} = s.{c}" for c in BUSINESS_KEY]
    )

    (
        target.alias("t")
        .merge(
            df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

### FULL REFRESH

def period_refresh(df: DataFrame) -> None:
    """
    Realiza carga completa da tabela.
    """

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RAW_TABLE)
    )

In [0]:
# Função de log de métricas do batch processado
# Registra contadores e estatísticas de processamento

def log_batch_metrics(batch_id: int) -> None:
    """
    Registra métricas básicas do micro-batch.
    """

    print("=" * 80)
    print(f"Batch ID........: {batch_id}")
    print("Persistência....: concluída")
    print(f"Tabela destino..: {RAW_TABLE}")
    print(f"Modo de carga...: {LOAD_MODE}")
    print("=" * 80)

In [0]:
# =============================================================================
# Execução do Pipeline
# =============================================================================

df_raw = read_sales_order()

# Capture metadata columns on the streaming DataFrame BEFORE entering foreachBatch
# (_metadata pseudo-columns are only available on streaming DataFrames, not on batch DataFrames)
df_with_metadata = df_raw.withColumns({
    "_source_file_name": F.col("_metadata.file_name"),
    "_source_file_path": F.col("_metadata.file_path"),
    "_source_file_modification_time": F.col("_metadata.file_modification_time"),
})

start_stream(df_with_metadata)

In [0]:
# Esta célula pode ser usada para verificar estatísticas da tabela após a carga
# Descomente e execute conforme necessário

# from pyspark.sql import functions as F
# 
# summary = (
#     spark.table(RAW_TABLE)
#          .groupBy("_load_id", "_ingested_by", "_ingested_at", "_source_file_name")
#          .agg(
#              F.count("*").alias("rows"),
#              F.countDistinct("numero_ov").alias("orders"),
#              F.min("data").alias("data_min"),
#              F.max("data").alias("data_max")
#          )
#          .withColumn(
#              "_ingested_at",
#              F.date_format(F.col("_ingested_at"), "yyyy/MM/dd HH:mm:ss")
#          )
#          .withColumn(
#              "data_min",
#              F.date_format(F.col("data_min"), "yyyy/MM/dd")
#          )
#          .withColumn(
#              "data_max",
#              F.date_format(F.col("data_max"), "yyyy/MM/dd")
#          )
#          .orderBy(F.col("_ingested_at").desc())
# )
# 
# display(summary)